In [7]:
import pandas as pd

In [10]:
#import dataset
dirty_data = pd.read_csv("Data/listings.csv")
dirty_data.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,3191,Malleson Garden Cottage,3754,Brigitte,NaN,Ward 57,-33.947620,18.475990,Entire home/apt,783.0,3,79,2024-06-18,0.56,1,11,4,NaN
1,15007,Blaauwberg House on the beach in Bloubergstrand,59072,Dirk,NaN,Ward 23,-33.800010,18.460630,Entire home/apt,6550.0,2,47,2024-10-19,0.35,3,46,2,NaN
2,15068,Grande Bay,59318,Linda,NaN,Ward 23,-33.788260,18.459400,Entire home/apt,3000.0,4,0,NaN,NaN,6,356,0,NaN
3,15077,Relaxed beach living in style,59342,Georg,NaN,Ward 4,-33.858356,18.490376,Private room,2165.0,2,7,2022-06-16,0.05,6,90,0,NaN
4,15199,Self catering apartment,59694,Alexa,NaN,Ward 115,-33.911150,18.412350,Entire home/apt,2500.0,14,2,2016-04-15,0.02,1,365,0,NaN


Run Basic Data Quality Checks

We need to identify 

1. Missing values
2. Duplicate Rows
3. Basic Data Characteristics


In [11]:
#create a function that can handle that 
def check_data_quality(dirty_data):
    #check for missing values
    missing_values = dirty_data.isnull().sum()
    #check for duplicates
    duplicates = dirty_data.duplicated().sum()
    #check for data types
    data_types = dirty_data.dtypes
    return missing_values, duplicates, data_types

In [12]:
print(check_data_quality(dirty_data))

(id                                    0
name                                  1
host_id                               0
host_name                            71
neighbourhood_group               25816
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                              4306
minimum_nights                        0
number_of_reviews                     0
last_review                        6358
reviews_per_month                  6358
calculated_host_listings_count        0
availability_365                      0
number_of_reviews_ltm                 0
license                           25732
dtype: int64, 0, id                                  int64
name                               object
host_id                             int64
host_name                          object
neighbourhood_group               float64
neighbourhood                      object
latitude  

In [19]:
def data_quality(dirty_data):
    #store initial data quality metrics
    quality_report = {
        "missing_values" : dirty_data.isnull().sum() /len(dirty_data) * 100,
        "duplicates" : dirty_data.duplicated().sum(),
        "total_rows":  len(dirty_data)
          }
    return quality_report

print(data_quality(dirty_data))

{'missing_values': id                                  0.000000
name                                0.003874
host_id                             0.000000
host_name                           0.275023
neighbourhood_group               100.000000
neighbourhood                       0.000000
latitude                            0.000000
longitude                           0.000000
room_type                           0.000000
price                              16.679579
minimum_nights                      0.000000
number_of_reviews                   0.000000
last_review                        24.628138
reviews_per_month                  24.628138
calculated_host_listings_count      0.000000
availability_365                    0.000000
number_of_reviews_ltm               0.000000
license                            99.674620
dtype: float64, 'duplicates': 0, 'total_rows': 25816}


In [20]:
dirty_data.dtypes

id                                  int64
name                               object
host_id                             int64
host_name                          object
neighbourhood_group               float64
neighbourhood                      object
latitude                          float64
longitude                         float64
room_type                          object
price                             float64
minimum_nights                      int64
number_of_reviews                   int64
last_review                        object
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
number_of_reviews_ltm               int64
license                            object
dtype: object

In [22]:
#convert the last_review column to datetime
def standardize_datatypes(dirty_data):
    dirty_data['last_review'] = pd.to_datetime(dirty_data['last_review'])
    return dirty_data.dtypes
print(standardize_datatypes(dirty_data))

id                                         int64
name                                      object
host_id                                    int64
host_name                                 object
neighbourhood_group                      float64
neighbourhood                             object
latitude                                 float64
longitude                                float64
room_type                                 object
price                                    float64
minimum_nights                             int64
number_of_reviews                          int64
last_review                       datetime64[ns]
reviews_per_month                        float64
calculated_host_listings_count             int64
availability_365                           int64
number_of_reviews_ltm                      int64
license                                   object
dtype: object


In [ ]:
#Handling missing values
from sklearn.impute import SimpleImputer

def handle_missing_values(dirty_data):
    #Handle numeric columns
    numeric_columns = dirty_data.select_dtypes(include=['int64', 'float64']).columns
    if len(numeric_columns) > 0:
        imputer = SimpleImputer(strategy='median')
        dirty_data[numeric_columns] = imputer.fit_transform(dirty_data[numeric_columns])
    
    #Handle categorical columns
    categorical_columns = dirty_data.select_dtypes(include=['object']).columns
    if len(categorical_columns) > 0:
        imputer = SimpleImputer(strategy='most_frequent')
        dirty_data[categorical_columns] = imputer.fit_transform(dirty_data[categorical_columns])
    return dirty_data




In [24]:
#remove outliers 
def remove_outliers(dirty_data):
    numeric_columns = dirty_data.select_dtypes(include=['int64', 'float64'].columns)
    outliers_removed = {}
    for col in numeric_columns:
        Q1 = dirty_data[col].quantile(0.25)
        Q3 = dirty_data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # Count outliers before removing
        outliers = len(dirty_data[(dirty_data[col] < lower_bound) | (dirty_data[col] > upper_bound)])
        
        # Cap the values instead of removing them
        dirty_data[col] = dirty_data[col].clip(lower=lower_bound, upper=upper_bound)
        
        if outliers > 0:
            outliers_removed[col] = outliers
            
    return dirty_data, outliers_removed

In [ ]:
#validate the results
def validate_cleaning(dirty_data, original_shape,cleaning_report):
    